### This file is corresponding to the second construction in our paper that is the RLWR MKFHE based on the FHE proposed in our work itself.

In [1]:
import numpy as np
from scipy import signal
from random import SystemRandom
import math
from math import log2, ceil
from time import time
import scipy
from random import SystemRandom
import numpy as np
from random import SystemRandom




"""
poly_multiplication.py implements a polynomial multiplication leveraging scipy's complex FFT implementation.
To multiply degree N polynomials f and g with integer coefficients one calls fast_poly_mult(f,g).
More details on how to use the complex FFT to multiply integer polynomials are given 
https://github.com/rtitiu/polymul-approx-ffts.
"""
def mod_vec(x_vec, modulus):
	x_vec = [int(i) for i in x_vec]
# 	print("x_vec: ", x_vec)
# 	print("modulus: ", modulus)
	log_modulus = ceil(log2(modulus))
	x_vec = np.array(x_vec, dtype = object)
	positive_r = np.bitwise_and(x_vec, modulus - 1)
	r_vec = positive_r - (modulus * (np.round(np.divide(positive_r, modulus).astype(float)).astype(int)))
	return r_vec

def poly_base_decomposition(f, B): 
	N = len(f)
	f = np.array(f, dtype = object)
	k = ceil( log2(2 * max(f) + 1) / log2(B))
	decomposed_f = np.zeros((N, k), dtype = object)
	for j in range(k):
		r = mod_vec(f, B)
		decomposed_f[:,j] = r 
		f = (f - r) // B
	return decomposed_f

def fast_mult(decomposed_f, decomposed_g, B = 2 ** 19):
	N = decomposed_f.shape[0]
	k_f = decomposed_f.shape[1]
	k_g = decomposed_g.shape[1]
	decomposed_fg = np.zeros((N + 1, k_f + k_g - 1), dtype = 'complex128')	

	for i in range(k_f):
		for j in range(k_g):
			decomposed_fg[:,i + j] += scipy.fft.rfft(decomposed_f[:, i], 2 * N) * scipy.fft.rfft(decomposed_g[:, j], 2 * N) 
	fg_recovered = np.array([0] * (2 * N), dtype = object)
	for j in reversed(range(k_f + k_g - 1)):
		rounded_term = scipy.fft.irfft(decomposed_fg[:,j], 2 * N)
		rounded_term = np.round(rounded_term.real).astype(int)
		fg_recovered *= B
		fg_recovered += rounded_term
	return fg_recovered	
	
def fast_poly_mult(f, g, base = 2 ** 19):
	f = np.array(f, dtype = 'object')
	g = np.array(g, dtype = 'object')
	f_dec = poly_base_decomposition(f, base)	
	g_dec = poly_base_decomposition(g, base)

	return fast_mult(f_dec, g_dec, base)


In [2]:


def uniform_vector(A,B): 
	'''
	Input : integers A,B
	Output: a vector of len N with uniform integer entries in range(A,B+1)  
	'''
	return [SystemRandom().randrange(B - A + 1) + A for _ in range(N)]

def round_vec(vec_x, pp, qq):
	'''
	Input : vec_x a vector of integers
	Output: nearest integer vector to vec_x * pp / qq
	'''
	vec_x = np.array(vec_x)
	return (2 * pp * vec_x + qq) // (2 * qq) 

def int2base(n, b):
	#Input : integer n and a base b
	#Output: a vector of digits corresponding to the decomposition of n in base b     
    if n < b:
        return [n]
    else:
        return [n % b] + int2base(n // b, b) 

def poly_add(p1, p2, modulus = None): 
	'''
	Input : np.arrays p1 and p2 of the same length
	Output: component-wise sum of the two vectors (reduced modulo 'modulus')
	'''
	p1 = np.array(p1, dtype = object)
	p2 = np.array(p2, dtype = object)
	addition = p1 + p2 
	if modulus == None:
		return addition
	else:	
		return np.array([x % modulus for x in addition], dtype = object)	

	'''
	Input : integer vectors p1, p2 representing the coefficients of polynomials of degree at most N - 1
	Output: integer vector representing the product polynoial p1 * p2 reduced modulo X^N + 1  
	'''  
def poly_mul(p1, p2):
    p1 = np.array(p1, dtype = object)
    p2 = np.array(p2, dtype = object)
    product = fast_poly_mult(p1, p2)
    product = np.concatenate( (product, [0] * ( 2 * N - len(product) )), None) 
    return np.array([int(product[i]) - int(product[i + N]) for i in range(N)], dtype = object)


def KeyGen():
	sk = np.array(uniform_vector(-1,1), dtype = object)
	a = np.array(uniform_vector(0,q - 1), dtype = object)
	b = round_vec(-1*poly_mul(a, sk), p1, q) % p1
	pk = (a, b)
	return (sk, pk)

def RelinKeyGen(sk):
    base = 2
    RelinKey = []
    l = ceil(log2(q/2)/ log2(base)) #k = ceil(log_base(q / 2))
    ss = poly_mul(sk, sk)
    for i in range(l):
        a_i = np.array(uniform_vector(0, p1 - 1), dtype = object)
        mask = round_vec(-1*poly_mul(a_i, sk), p2, p1) % p2
        r1 = (mask + base ** i * ss) % p2
        RelinKey.append((r1,a_i))
    return RelinKey

def Encrypt(message, pub_key):
    (a,b) = pub_key
    rnd = uniform_vector(-1,1)
    c0 = round_vec(poly_mul(b, rnd), p2, p1) % p2
    encoded_message = Delta * np.array(message, dtype = object)
    c0 = poly_add(c0, encoded_message) % p2
    c1 = round_vec(poly_mul(a, rnd), p2, q) % p2
    return (c0,c1)

def Decrypt(ct, sk):
	(c0,c1) = ct
	# c0 = np.array(c0, dtype = object)
	# c1 = np.array(c1, dtype = object)
	# sk = np.array(sk, dtype = object)
	c1sk = poly_mul(c1,sk)%p2
	# scaled_c1 = q * c1	
	return round_vec(c0 + c1sk, t, p2) % t

def CiphertextAddition(ct1, ct2):
	return ((ct1[0] + ct2[0]) % p2, (ct1[1] + ct2[1]) % p2)

def CiphertextMultiplication(ct1, ct2, rkey):
    base = 2 
    l = ceil(log2(q/2)/ log2(base))
    c0 = round_vec(poly_mul(ct1[0], ct2[0]), t, p2) % p2
    c1 = round_vec(poly_mul(ct1[0], ct2[1]) + poly_mul(ct1[1], ct2[0]), t, p2) % p2
    c2 = round_vec(poly_mul(ct1[1], ct2[1]), t, p2) % p2
    
    decomposed_c2 = np.zeros((N, l), dtype = object)	
    for i in range(N):
        into_base = int2base(c2[i],base)
        decomposed_c2[i] = np.array(into_base + [0] * (l - len(into_base)), dtype = object)
    
    v = c0
    w = np.array([0]*N, dtype=object)
    for j in range(l):
        v = (v + poly_mul(rkey[j][0], decomposed_c2[:,j])) % p2
        w = (w + poly_mul(rkey[j][1], decomposed_c2[:,j])) % p1
    
    w = (c1 + round_vec(w, p2, p1)) % p2
    return (v,w)	




### Testing

In [3]:
# print("N = {}, q = {}, t = {}".format(N, q, t))
# t0 = time()	
# (sk, pk) = KeyGen()
# t1 = time()
# print("KeyGen:        {:.2f}s".format(t1-t0)) 

# t11 = time()
# rkey = RelinKeyGen(sk)
# t2 = time()
# print("Relin Keygen:  {:.2f}s".format(t2 - t11))

# msg1 = uniform_vector(-t//2, t - t//2 -1)
# msg2 = uniform_vector(-t//2, t - t//2 -1)

# t3 = time()
# ct1 = Encrypt(msg1, pk)
# t4 = time()
# print("Encryption:    {:.2f}s".format(t4 - t3))
# ct2 = Encrypt(msg2, pk)

# t9 = time()
# Decrypt(ct1,sk)
# t10 = time()
# print("Decryption:    {:.2f}s".format(t10 - t9))

# t5 = time()
# ct1_plus_ct2 = CiphertextAddition(ct1, ct2)
# t6 = time()
# print("Ct_addition:   {:.2f}s".format(t6 - t5))

# t7 = time()
# ct_multiplied = CiphertextMultiplication(ct1, ct2, rkey)
# t8 = time()
# print("Ct_mult:       {:.2f}s (relin time is included)".format(t8 - t7))

# print(np.array_equal(Decrypt(ct_multiplied, sk), poly_mul(msg1, msg2) % t))

### Testing FHE

In [4]:
# import math
# parameter_sets = [
   
# {"N": 2**13, "q": 2**209,  "p1": 2**205,  "p2": 2**201},
# {"N": 2**14, "q": 2**428,  "p1": 2**424,  "p2": 2**420},

# ]


# attempts = 3

# for params in parameter_sets:
#     N = params["N"]
#     q = params["q"]
#     p1 = params["p1"]
#     p2 = params["p2"]
    
#     print("testing for parameter sets N ={}, log2(q)={}, log2(p1)={}, log2(p2)={}".format(N, log2(q), log2(p1), log2(p2)))
#     Delta = p2 // t


#     (sk, pk) = KeyGen()
#     rkey = RelinKeyGen(sk)

    
#     for trail in range(attempts):
#         msg1 = np.array(uniform_vector(-t//2, t - t//2 -1), dtype = object) % t
#         ct1 = Encrypt(msg1, pk)
#         avg = 0
#         i = 0
#         while(True):
#             msg2 = np.array(uniform_vector(-t//2, t - t//2 -1), dtype = object) % t
#             ct2 = Encrypt(msg2, pk)
#             ct_multiplied = CiphertextMultiplication(ct1, ct2, rkey)
#             mesg_mul = poly_mul(msg1, msg2) % t
#             if not(np.array_equal(Decrypt(ct_multiplied, sk), mesg_mul)):
#                 print("failure")
#                 break
#             print("multiplicative depth: {}".format(i))
#             i+=1
#             msg1 = mesg_mul
#             ct1 = ct_multiplied
#         avg+=i
#         print("depth: {}".format(i))
#     print("The average depth for this parameter set: {}".format(avg))
#     print("------------------------------------------------------------")

## MKFHE from RLWR (Construction 2)

In [5]:
def get_error(del_r, k,d,l, s_i=1, w=2, security_level=128, modulo_diff_power=4):
    
    """
    Input:  - del_r: the minimum degree of the polynomial.
            - k: the number of the parties.
            - d: the circuit depth to be evaluated.
            - s_i: the norm of the secret for one party.
            - l: the log2 of the moduli to start from.
            - w : the decomposition base.
            - modulo_diff_power: the difference between powers (in our case 4) 
    Output: the parameter set that matches the required level of security.
    """
    s_norm = k*s_i
    p2byp1 = 2**(-1*modulo_diff_power)

  
    l = l-(modulo_diff_power*2)
    y_in1 = (p2byp1 + 1)*(del_r*s_norm)/2 + 1/2
    c_11 = (del_r*s_norm + 4)*(del_r*t) + del_r/2
    c_21= (t**2)*del_r*(del_r*s_norm/2 + 2.5) + ((del_r**2 )*(s_norm**2)+ (del_r*s_norm) + 1)/2 + (3*l*w*k*(del_r**2)*s_norm*p2byp1)/4 + k*l*w*del_r/2 + del_r*s_norm/2
    d_e2 = (c_11**(d-1))*(c_11*y_in1 + c_21)
    return round(math.log2(d_e2/k))


In [6]:
def KeyGen_multiparty(k):
    a = np.array(uniform_vector(0,q - 1), dtype=object)
    pk_list = []
    sk_list = []
    for i in range(k):
        sk = np.array(uniform_vector(-1,1), dtype=object)
        # print("sk len: ", len(sk))
        b = round_vec(-1*poly_mul(a, sk), p1,q) % p1
        pk = (a, b)
        sk_list.append(sk)
        pk_list.append(pk)
    return (sk_list, pk_list)

def KeyExt(pk_list):
    a = pk_list[0][0]
    b = pk_list[0][1]

    for i in range (1, len(pk_list)):
        b = (b + pk_list[i][1]) % p1

    return (a, b) 
            
def RelinKeyGen_multiparty(sk_list):
    base = 2
    k_parties = len(sk_list)

    l = ceil(log2(q) / log2(base)) 
    
    a_r = [np.array(uniform_vector(0, q - 1), dtype=object) for _ in range(l)]
    w_vec = [base ** j for j in range(l)]
    
    u_list = []
    h0_i_list = []
    h1_i_list = []
    
    # Step 1
    for i in range(k_parties):
        s_i = sk_list[i]
        
        u_i = np.array(uniform_vector(-1, 1), dtype=object)
        u_list.append(u_i)
        
        h0_i = []
        h1_i = []        
        
        for j in range(l):
            term1_0 = round_vec(-1*poly_mul(u_i, a_r[j]), p1,q) % p1
            term2_0 = round_vec(s_i * w_vec[j], p1, p2) % p1            
            h0_i.append((term1_0 + term2_0) % p1)
            
            term1_1 = round_vec(poly_mul(s_i, a_r[j]), p1, q) % p1
            h1_i.append(term1_1) 
            
        h0_i_list.append(h0_i)
        h1_i_list.append(h1_i)
        
    h0 = np.sum(h0_i_list, axis=0) % p1
    h1 = np.sum(h1_i_list, axis=0) % p1

    # print(h0)
    
    # Step 2
    hp0_i_list = []
    hp1_i_list = []
    
    for i in range(k_parties):
        s_i = sk_list[i]
        u_i = u_list[i]
        
        # s_minus_u = s_i - u_i 
        u_minus_s = u_i - s_i 
        
        hp0_i = []
        hp1_i = []
        
        for j in range(l):
            hp0_val = round_vec(poly_mul(s_i, h0[j]), p2, p1) % p2
            hp0_i.append(hp0_val)
            
            hp1_val = round_vec(poly_mul(u_minus_s, h1[j]), p2, p1) % p2
            hp1_i.append(hp1_val)
            
        hp0_i_list.append(hp0_i)
        hp1_i_list.append(hp1_i)
        
    hp0 = np.sum(hp0_i_list, axis=0) % p2
    hp1 = np.sum(hp1_i_list, axis=0) % p2
    
    r0 = (hp0 + hp1) % p2
    r1 = h1
    
    # return (r0, r1)
    return list(zip(r0, r1))


def Encrypt_multiparty(message, epk):
    (a,b) = epk
    rnd = uniform_vector(-1,1)
    c0 = round_vec(poly_mul(b, rnd), p2, p1) % p2
    encoded_message = Delta * np.array(message, dtype = object)
    c0 = poly_add(c0, encoded_message) % p2
    c1 = round_vec(poly_mul(a, rnd), p2, q) % p2
    return (c0,c1)
            

def Decrypt_multiparty(sk_list, ct, lamda=128): ###lamba is passed to be lambda+circuit error
    (c0, c1) = ct
    k_parties = len(sk_list)
    
    # p_i_list = []
    lamda = lamda-int(math.log2(k_parties)) #### divide by k to get the exact value per each party
    sum_p_i = np.array(uniform_vector(0, 0), dtype = object)
    
    # Partial Decryption
    for i in range(k_parties):
        s_i = sk_list[i]
        e_sm = np.array(uniform_vector(0, 2**lamda), dtype = object) 
        
        mul_term = poly_mul(c1, s_i) % p2    
        p_i = (mul_term + e_sm) % p2
        sum_p_i=(sum_p_i+p_i) % p2
        # p_i_list.append(p_i)
        
    # Final Decryption
    # sum_p_i = np.sum(p_i_list, axis=0) % q
    
    # scaled_sum = round_vec(sum_p_i, p, q)
    
    noisy_m = (c0 + sum_p_i) % p2
    m = round_vec(noisy_m, t, p2) % t
    
    return m

def CiphertextMultiplication_multiparty(ct1, ct2, rkey):
    base = 2 
    l = ceil(log2(q)/ log2(base))
    c0 = round_vec(poly_mul(ct1[0], ct2[0]), t, p2) % p2
    c1 = round_vec(poly_mul(ct1[0], ct2[1]) + poly_mul(ct1[1], ct2[0]), t, p2) % p2
    c2 = round_vec(poly_mul(ct1[1], ct2[1]), t, p2) % p2
    
    decomposed_c2 = np.zeros((N, l), dtype = object)	
    for i in range(N):
        into_base = int2base(c2[i],base)
        decomposed_c2[i] = np.array(into_base + [0] * (l - len(into_base)), dtype = object)
    
    v = c0
    w = np.array([0]*N, dtype=object)
    for j in range(l):
        v = (v + poly_mul(rkey[j][0], decomposed_c2[:,j])) % p2
        w = (w + poly_mul(rkey[j][1], decomposed_c2[:,j])) % p1
    
    w = (c1 + round_vec(w, p2, p1)) % p2
    return (v,w)	



### Testing for some parameter sets from our paper (checking the actual circuit depth matches those selected in the parametersets)

In [ ]:
attempts = 10
k_parties = 2 

parameter_sets = [
   
{"N": 2**13, "q": 2**209,  "p1": 2**205,  "p2": 2**201},
{"N": 2**14, "q": 2**428,  "p1": 2**424,  "p2": 2**420},
{"N": 2**15, "q": 2**871, "p1": 2**867, "p2": 2**863},
{"N": 2**16, "q": 2**1746, "p1": 2**1742, "p2": 2**1738},
]



for params in parameter_sets:
    N = params["N"]
    q = params["q"]
    p1 = params["p1"]
    p2 = params["p2"]
    
    print("testing for parameter sets N ={}, log2(q)={}, log2(p1)={}, log2(p2)={}".format(
        N, log2(q), log2(p1), log2(p2)))
    
    t = 3
    Delta = p2 // t

    (sk_list, pk_list) = KeyGen_multiparty(k_parties)
    
    epk = KeyExt(pk_list)
    
    rkey = RelinKeyGen_multiparty(sk_list)

    avg = 0
    
    for trial in range(attempts):
        msg1 = np.array(uniform_vector(-1,1), dtype=object) % t
        ct1 = Encrypt_multiparty(msg1, epk)
        
        i = 0
        while(True):
            msg2 = np.array(uniform_vector(-1,1), dtype=object) % t
            ct2 = Encrypt_multiparty(msg2, epk)
            
            ct_multiplied = CiphertextMultiplication_multiparty(ct1, ct2, rkey)
            
            mesg_mul = poly_mul(msg1, msg2) % t
            error_magnitude = get_error(N, k_parties,(i+1),math.log2(q), s_i=1, w=2, security_level=128, modulo_diff_power=4)
            decrypted_msg = Decrypt_multiparty(sk_list, ct_multiplied,lamda=(128+error_magnitude))

            print("mesg_mul: ", mesg_mul)
            print("decrypted_msg: ", decrypted_msg)
            
            if not(np.array_equal(decrypted_msg, mesg_mul % t)):
                print("failure") 
                break
                
            print("multiplicative depth: {}".format(i))
            i += 1
            msg1 = mesg_mul
            ct1 = ct_multiplied
            
        avg += i
        print("depth: {}".format(i))
        
    print("The average depth for this parameter set: {}".format(avg / attempts)) 
    print("------------------------------------------------------------")

testing for parameter sets N =8192, log2(q)=209.0, log2(p1)=205.0, log2(p2)=201.0
mesg_mul:  [1 2 2 ... 2 1 1]
decrypted_msg:  [1 2 2 ... 2 1 1]
multiplicative depth: 0
mesg_mul:  [1 1 1 ... 2 2 1]
decrypted_msg:  [1 1 1 ... 2 2 1]
multiplicative depth: 1
mesg_mul:  [2 2 0 ... 1 0 1]
decrypted_msg:  [0 2 0 ... 0 2 1]
failure
depth: 2
mesg_mul:  [0 1 2 ... 1 1 0]
decrypted_msg:  [0 1 2 ... 1 1 0]
multiplicative depth: 0
mesg_mul:  [0 2 1 ... 0 2 1]
decrypted_msg:  [0 2 1 ... 0 2 1]
multiplicative depth: 1
mesg_mul:  [2 0 1 ... 1 1 0]
decrypted_msg:  [1 2 0 ... 2 1 1]
failure
depth: 2
mesg_mul:  [2 1 0 ... 0 2 2]
decrypted_msg:  [2 1 0 ... 0 2 2]
multiplicative depth: 0
mesg_mul:  [1 0 0 ... 2 1 1]
decrypted_msg:  [1 0 0 ... 2 1 1]
multiplicative depth: 1
mesg_mul:  [2 2 2 ... 0 2 1]
decrypted_msg:  [2 2 2 ... 1 0 1]
failure
depth: 2
mesg_mul:  [0 2 0 ... 2 2 1]
decrypted_msg:  [0 2 0 ... 2 2 1]
multiplicative depth: 0
mesg_mul:  [1 0 0 ... 2 0 2]
decrypted_msg:  [1 0 0 ... 2 0 2]
multip